In [1]:
import pandas as pd
import numpy as np

# Clases propias
from etl import Dataloader
from feature_engineer import add_features
from train import Train
from train_with_mlflow import TrainWithMLflow

# Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb
from xgboost import XGBClassifier

# MLflow
import mlflow


c:\Users\Valentina Molina\Documents\repositorios\Proyecto_Final_MLOps\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ETL

In [2]:
loader = Dataloader(
    path_to_save=r"C:/Users/Valentina Molina/Documents/Repositorios/Proyecto_Final_MLOps/data/PS_20174392719_1491204439457_log.csv",
    n_samples=50000
)
df = loader.load_data()
loader.drop_name_columns()
df = loader.df.copy()

print(df.shape)
df.head()

(50000, 8)


,step,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud
0,1,PAYMENT,9839.64,170136.0,160296.36,0.0,0.0,0
1,1,PAYMENT,1864.28,21249.0,19384.72,0.0,0.0,0
2,1,TRANSFER,181.00,181.0,0.00,0.0,0.0,1
3,1,CASH_OUT,181.00,181.0,0.00,21182.0,0.0,1
4,1,PAYMENT,11668.14,41554.0,29885.86,0.0,0.0,0


Feature Engineering

In [3]:
df = add_features(df)
df.head()

,step,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,diff_old_new_orig,diff_old_new_dest,amount_to_orig_balance,amount_to_dest_balance
0,1,PAYMENT,9839.64,170136.0,160296.36,0.0,0.0,0,9839.64,0.0,0.057834,9839.640000
1,1,PAYMENT,1864.28,21249.0,19384.72,0.0,0.0,0,1864.28,0.0,0.087731,1864.280000
2,1,TRANSFER,181.00,181.0,0.00,0.0,0.0,1,181.00,0.0,0.994505,181.000000
3,1,CASH_OUT,181.00,181.0,0.00,21182.0,0.0,1,181.00,21182.0,0.994505,0.008545
4,1,PAYMENT,11668.14,41554.0,29885.86,0.0,0.0,0,11668.14,0.0,0.280788,11668.140000


Definir variables

In [4]:
numeric_features = [
    'step', 'amount',
    'oldbalanceOrg', 'newbalanceOrig',
    'oldbalanceDest', 'newbalanceDest',
    'diff_old_new_orig', 'diff_old_new_dest',
    'amount_to_orig_balance', 'amount_to_dest_balance'
]

categorical_features = ['type']
target_column = 'isFraud'
test_size = 0.2


Step 1: Modelando sin MLflow

In [5]:
# Logistic Regression
model = LogisticRegression(max_iter=1000, random_state=42)
trainer = Train(df, numeric_features, categorical_features, target_column, model, test_size)
pipeline = trainer.train()
print("Modelo Logistic Regression entrenado sin MLflow ✅")

# Random Forest
model = RandomForestClassifier(n_estimators=100, random_state=42)
trainer = Train(df, numeric_features, categorical_features, target_column, model, test_size)
pipeline = trainer.train()
print("Modelo RandomForest entrenado sin MLflow ✅")


Modelo Logistic Regression entrenado sin MLflow ✅
Modelo RandomForest entrenado sin MLflow ✅


Step 2: Modelando con MLflow (Logistic + RandomForest + LightGBM)

In [6]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("fraude-modelos")

mlflow.sklearn.autolog()

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb

modelos = [
    ("LogisticRegression", LogisticRegression(max_iter=1000, random_state=42)),
    ("RandomForest", RandomForestClassifier(n_estimators=100, random_state=42)),
    ("LightGBM", lgb.LGBMClassifier(random_state=42))
]

In [7]:
resultados = {}
for nombre, modelo in modelos:
    print(f"\nEntrenando {nombre} con MLflow...")
    trainer = TrainWithMLflow(df, numeric_features, categorical_features, target_column, modelo, test_size)
    pipeline, run_id = trainer.train()
    resultados[nombre] = run_id

resultados


Entrenando LogisticRegression con MLflow...
MLflow Run ID: 9f3ea05123e24a04bf4d523621f1c8c1
Tracking URI: http://127.0.0.1:5000
Train Accuracy: 0.9984
Test Accuracy: 0.9978
🏃 View run popular-sow-876 at: http://127.0.0.1:5000/#/experiments/729808979030306957/runs/9f3ea05123e24a04bf4d523621f1c8c1
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/729808979030306957

Entrenando RandomForest con MLflow...
MLflow Run ID: 9a87f2d2fff5458fa0731f29ce49dee7
Tracking URI: http://127.0.0.1:5000
Train Accuracy: 1.0000
Test Accuracy: 0.9997
🏃 View run debonair-snail-974 at: http://127.0.0.1:5000/#/experiments/729808979030306957/runs/9a87f2d2fff5458fa0731f29ce49dee7
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/729808979030306957

Entrenando LightGBM con MLflow...
[LightGBM] [Info] Number of positive: 80, number of negative: 39920
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.004894 seconds.
You can set `force_col_wise=true` to remove th

{'LogisticRegression': '9f3ea05123e24a04bf4d523621f1c8c1',
 'RandomForest': '9a87f2d2fff5458fa0731f29ce49dee7',
 'LightGBM': 'f26493cd72ce4cf99b1a1253d8d0160e'}

Modelo con XGBoost + MLflow

In [8]:
mlflow.set_experiment("fraude-xgboost")
mlflow.xgboost.autolog()

params_xgb = {
    "n_estimators": 100,
    "max_depth": 6,
    "learning_rate": 0.1,
    "subsample": 0.8
}

model_xgb = XGBClassifier(**params_xgb, random_state=42, use_label_encoder=False, eval_metric="logloss")

trainer = TrainWithMLflow(df, numeric_features, categorical_features, target_column, model_xgb, test_size, params_xgb, mlflow)
pipeline, run_id = trainer.train()
print("Modelo XGBoost entrenado con MLflow ✅", run_id)


MLflow Run ID: 275a9ce0ba55471abfb6f2fcd06edeea
Tracking URI: http://127.0.0.1:5000
Train Accuracy: 1.0000
Test Accuracy: 0.9994
🏃 View run bold-robin-160 at: http://127.0.0.1:5000/#/experiments/432587766316071821/runs/275a9ce0ba55471abfb6f2fcd06edeea
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/432587766316071821
Modelo XGBoost entrenado con MLflow ✅ 275a9ce0ba55471abfb6f2fcd06edeea


Step 3: Modelo con MLflow + Optuna

In [9]:
from train_with_mlflow_optuna import TrainWithMLflowOptuna

mlflow.set_experiment("fraude-optuna")

param_distributions = {
    'n_estimators': ('int', 50, 200),
    'max_depth': ('int', 5, 30),
    'min_samples_split': ('int', 2, 10),
    'min_samples_leaf': ('int', 1, 5),
    'max_features': ('categorical', ['sqrt', 'log2', None])
}

trainer = TrainWithMLflowOptuna(
    df=df,
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    target_column=target_column,
    model_class=RandomForestClassifier,
    test_size=0.2,
    n_trials=20,
    optimization_metric='f1',
    param_distributions=param_distributions,
    model_params={'random_state': 42},
    mlflow_setup=mlflow
)

best_pipeline, run_id, study = trainer.train()
print("Mejor modelo con Optuna:", run_id)


2025/09/22 19:19:55 INFO mlflow.tracking.fluent: Experiment with name 'fraude-optuna' does not exist. Creating a new experiment.
[I 2025-09-22 19:19:55,630] A new study created in memory with name: no-name-25d89609-9aef-46db-b4df-3da7b95d537a
2025/09/22 19:19:55 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '7bb5bb5bf9c84045bc9b9d9c2893c9f1', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow


🏃 View run unleashed-carp-945 at: http://127.0.0.1:5000/#/experiments/438737474763045644/runs/7bb5bb5bf9c84045bc9b9d9c2893c9f1
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/438737474763045644


[I 2025-09-22 19:20:23,866] Trial 0 finished with value: 0.918918918918919 and parameters: {'n_estimators': 178, 'max_depth': 25, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'log2'}. Best is trial 0 with value: 0.918918918918919.
2025/09/22 19:20:24 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID 'aed7621db02847399b4093176186c509', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow


🏃 View run receptive-mare-860 at: http://127.0.0.1:5000/#/experiments/438737474763045644/runs/aed7621db02847399b4093176186c509
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/438737474763045644


[I 2025-09-22 19:20:50,566] Trial 1 finished with value: 0.8888888888888888 and parameters: {'n_estimators': 199, 'max_depth': 9, 'min_samples_split': 2, 'min_samples_leaf': 3, 'max_features': 'log2'}. Best is trial 0 with value: 0.918918918918919.
2025/09/22 19:20:50 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '6fbdae4ae72742c3ae03aa94f921873b', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow


...

🏃 View run nebulous-hound-834 at: http://127.0.0.1:5000/#/experiments/438737474763045644/runs/8587bd062e964d78becc79ec39838386
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/438737474763045644


[I 2025-09-22 19:30:56,212] Trial 18 finished with value: 0.8888888888888888 and parameters: {'n_estimators': 180, 'max_depth': 20, 'min_samples_split': 5, 'min_samples_leaf': 4, 'max_features': 'log2'}. Best is trial 0 with value: 0.918918918918919.
2025/09/22 19:30:56 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '6545a5bfca8245b6a439f92d59885c1e', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow


🏃 View run wistful-shrimp-428 at: http://127.0.0.1:5000/#/experiments/438737474763045644/runs/6545a5bfca8245b6a439f92d59885c1e
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/438737474763045644


[I 2025-09-22 19:32:09,085] Trial 19 finished with value: 0.7222222222222222 and parameters: {'n_estimators': 123, 'max_depth': 24, 'min_samples_split': 9, 'min_samples_leaf': 1, 'max_features': None}. Best is trial 0 with value: 0.918918918918919.
2025/09/22 19:32:09 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '418eef99c4cb49959f19a77bea1d4bb0', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow


🏃 View run gifted-shrew-11 at: http://127.0.0.1:5000/#/experiments/438737474763045644/runs/418eef99c4cb49959f19a77bea1d4bb0
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/438737474763045644
Optuna best params: {'random_state': 42, 'n_estimators': 178, 'max_depth': 25, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'log2'}
MLflow run_id: 79b2e99a85dc463e99a617ed18e3eff1
🏃 View run auspicious-rook-185 at: http://127.0.0.1:5000/#/experiments/438737474763045644/runs/79b2e99a85dc463e99a617ed18e3eff1
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/438737474763045644
Mejor modelo con Optuna: 79b2e99a85dc463e99a617ed18e3eff1
